# RetinAI CNN v2 — especialista de glaucoma RIM-ONE

Entrena un detector binario real de glaucoma respetando la partición **por hospital** de RIM-ONE DL. Solo `training_set` se divide para entrenamiento/validación; `test_set` permanece intacto hasta la evaluación final.

El resultado es académico y no constituye aprobación clínica. Ejecuta todo con GPU T4.

In [ ]:
%pip -q install gdown scikit-learn importlib-resources
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, gdown, shutil

destino = Path('/content/drive/MyDrive/RETINAI_CNN_V2')
destino.mkdir(parents=True, exist_ok=True)
script_drive = destino / 'entrenar_glaucoma_rim_one_academico.py'
if not script_drive.is_file():
    raise RuntimeError(f'Falta el script en Drive: {script_drive}')
script = Path('/content/entrenar_glaucoma_rim_one_academico.py')
shutil.copy2(script_drive, script)
zip_rim = Path('/content/rim_one_dl_images.zip')
if not zip_rim.is_file():
    gdown.download(id='1teYi_smpLiNZNJcTWdxXgKKLW2fkUQr4', output=str(zip_rim), quiet=False)
sha = hashlib.file_digest(zip_rim.open('rb'), 'sha256').hexdigest()
esperado = '85aed5f95c794f52d11b6ed953032108a66576ffcff4b587365df467491b603d'
if sha != esperado:
    raise RuntimeError(f'El ZIP cambió: {sha}')
print('GPU:', __import__('tensorflow').config.list_physical_devices('GPU'))
print('RIM-ONE verificado:', zip_rim)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(script), str(zip_rim), str(destino)], check=True)
print('Modelo:', destino / 'glaucoma_rim_one_academico.tflite')
print('Informe:', destino / 'informe_glaucoma_rim_one_academico.json')


## Lectura del informe

La métrica principal para la exposición es `hospital_test`, porque esas 174 imágenes no participan en el ajuste del modelo ni del umbral. Compara sensibilidad, especificidad y AUC con el intento anterior, que detectó 0/56 glaucomas externos.